In [26]:
import copy
import json
import os
import random
from pathlib import Path
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import pyreadstat
import sklearn
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import flwr as fl
from flwr.simulation import start_simulation

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


## Load and audit the original NDHS data

In [27]:
# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------

DATA_PATH = Path(r"c:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA")

OUTPUT_DIR = Path("NDHS_FL_Research")
AUDIT_DIR = OUTPUT_DIR / "01_data_audit"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 2. LOAD ORIGINAL DATA
# ------------------------------------------------------------

df, meta = pyreadstat.read_dta(
    str(DATA_PATH),
    apply_value_formats=False
)

print("DATASET LOADED")
print("=" * 60)
print("Dataset shape:", df.shape)
print("Number of observations:", len(df))
print("Number of variables:", len(df.columns))

# ------------------------------------------------------------
# 3. REQUIRED VARIABLE CHECK
# ------------------------------------------------------------

required_variables = [
    "m15_1",
    "v024",
    "v001",
    "v021",
    "v005",
    "v022",
    "v023",
    "v501",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01",
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131",
    "b3_01",
    "v011"
]

missing_variables = [
    col for col in required_variables
    if col not in df.columns
]

if missing_variables:
    raise ValueError(
        f"Required variables missing from dataset: {missing_variables}"
    )

print("\nAll required variables are present.")

# ------------------------------------------------------------
# 4. RECONSTRUCT THE BINARY OUTCOME
# ------------------------------------------------------------

facility_codes = [21, 22, 23, 26, 31, 36]
home_codes = [11, 12]

df["m15_1"] = pd.to_numeric(df["m15_1"], errors="coerce")

df["facility_delivery"] = np.select(
    [
        df["m15_1"].isin(home_codes),
        df["m15_1"].isin(facility_codes)
    ],
    [
        0,
        1
    ],
    default=np.nan
)

# Keep only observations with a valid binary outcome.
model_df = df.loc[
    df["facility_delivery"].notna()
].copy()

model_df["facility_delivery"] = (
    model_df["facility_delivery"].astype(int)
)

print("\nOUTCOME RECONSTRUCTION")
print("=" * 60)

print("Original observations:", len(df))
print("Eligible observations:", len(model_df))
print("Excluded observations:", len(df) - len(model_df))

print("\nOutcome distribution:")
print(
    model_df["facility_delivery"]
    .value_counts()
    .sort_index()
)

print("\nOutcome percentages:")
print(
    model_df["facility_delivery"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

# ------------------------------------------------------------
# 5. CHECK REGIONAL DISTRIBUTION
# ------------------------------------------------------------

region_summary = (
    model_df.groupby("v024")
    .agg(
        sample_size=("facility_delivery", "size"),
        facility_deliveries=("facility_delivery", "sum"),
        facility_prevalence=("facility_delivery", "mean")
    )
    .reset_index()
)

region_summary["facility_prevalence"] *= 100

print("\nREGIONAL DISTRIBUTION")
print("=" * 60)
print(region_summary.to_string(index=False))

region_summary.to_csv(
    AUDIT_DIR / "regional_outcome_distribution.csv",
    index=False
)

# ------------------------------------------------------------
# 6. VERIFY PSU AND CLUSTER IDENTIFIERS
# ------------------------------------------------------------

print("\nPSU AND CLUSTER CHECKS")
print("=" * 60)

print("v001 missing:", df["v001"].isna().sum())
print("v021 missing:", df["v021"].isna().sum())

same_cluster = (
    df["v001"].astype("string")
    == df["v021"].astype("string")
)

print("v001 equals v021 for all nonmissing pairs:",
      same_cluster[df["v001"].notna() & df["v021"].notna()].all())

print("Unique v001:", df["v001"].nunique())
print("Unique v021:", df["v021"].nunique())

print(
    "Unique PSUs in eligible sample:",
    model_df["v021"].nunique()
)

# Check whether a PSU is assigned to more than one region.
psu_zone_counts = (
    model_df.groupby("v021")["v024"]
    .nunique()
)

print(
    "PSUs appearing in multiple regions:",
    int((psu_zone_counts > 1).sum())
)

# ------------------------------------------------------------
# 7. SURVEY WEIGHT AND STRATA CHECKS
# ------------------------------------------------------------

model_df["weight"] = (
    pd.to_numeric(model_df["v005"], errors="coerce") / 1_000_000
)

print("\nSURVEY DESIGN CHECKS")
print("=" * 60)

print("Missing weights:", model_df["weight"].isna().sum())
print("Non-positive weights:", (model_df["weight"] <= 0).sum())
print("Unique PSUs:", model_df["v021"].nunique())
print("Unique v022 strata:", model_df["v022"].nunique())
print("Unique v023 values:", model_df["v023"].nunique())

if model_df["weight"].isna().any():
    raise ValueError("Missing survey weights found in eligible sample.")

if (model_df["weight"] <= 0).any():
    raise ValueError("Non-positive survey weights found.")

# Weighted and unweighted outcome prevalence.
unweighted_prevalence = model_df["facility_delivery"].mean()

weighted_prevalence = np.average(
    model_df["facility_delivery"],
    weights=model_df["weight"]
)

print("\nUnweighted facility prevalence:",
      round(unweighted_prevalence * 100, 3), "%")

print("Weighted facility prevalence:",
      round(weighted_prevalence * 100, 3), "%")

# Regional weighted prevalence.
weighted_region_rows = []

for zone, group in model_df.groupby("v024"):

    weighted_rate = np.average(
        group["facility_delivery"],
        weights=group["weight"]
    )

    weighted_region_rows.append({
        "v024": zone,
        "sample_size": len(group),
        "unweighted_prevalence": group["facility_delivery"].mean(),
        "weighted_prevalence": weighted_rate
    })

weighted_region_summary = pd.DataFrame(weighted_region_rows)

weighted_region_summary[
    ["unweighted_prevalence", "weighted_prevalence"]
] *= 100

weighted_region_summary.to_csv(
    AUDIT_DIR / "weighted_regional_prevalence.csv",
    index=False
)

print("\nWeighted regional prevalence:")
print(weighted_region_summary.to_string(index=False))

# ------------------------------------------------------------
# 8. INSPECT v701 VALUE LABELS AND MISSINGNESS
# ------------------------------------------------------------

print("\nPARTNER EDUCATION: v701")
print("=" * 60)

print("Variable label:")
print(meta.column_names_to_labels.get("v701", "No label available"))

print("\nValue labels:")
print(meta.variable_value_labels.get("v701", "No value labels available"))

print("\nObserved v701 frequencies, including missing:")
print(
    model_df["v701"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMissing v701 by marital status (v501):")

v701_missing_by_marital = (
    model_df.assign(v701_missing=model_df["v701"].isna())
    .groupby("v501", dropna=False)
    .agg(
        sample_size=("v701_missing", "size"),
        missing_v701=("v701_missing", "sum")
    )
    .reset_index()
)

v701_missing_by_marital["missing_percentage"] = (
    100 * v701_missing_by_marital["missing_v701"]
    / v701_missing_by_marital["sample_size"]
)

print(v701_missing_by_marital.to_string(index=False))

v701_missing_by_marital.to_csv(
    AUDIT_DIR / "v701_missingness_by_marital_status.csv",
    index=False
)

# ------------------------------------------------------------
# 9. FEATURE MISSINGNESS AUDIT
# ------------------------------------------------------------

baseline_features = [
    "age_at_birth",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01"
]

expanded_features = baseline_features + [
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131"
]

# Derive age at birth.
model_df["age_at_birth"] = np.floor(
    (pd.to_numeric(model_df["b3_01"], errors="coerce")
     - pd.to_numeric(model_df["v011"], errors="coerce")) / 12
)

feature_missingness = pd.DataFrame({
    "feature": expanded_features,
    "missing_n": [
        model_df[col].isna().sum()
        for col in expanded_features
    ],
    "missing_percentage": [
        model_df[col].isna().mean() * 100
        for col in expanded_features
    ]
})

print("\nFEATURE MISSINGNESS")
print("=" * 60)
print(feature_missingness.to_string(index=False))

feature_missingness.to_csv(
    AUDIT_DIR / "feature_missingness.csv",
    index=False
)

# ------------------------------------------------------------
# 10. SAVE AUDIT SUMMARY
# ------------------------------------------------------------

audit_summary = {
    "source_file": DATA_PATH.name,
    "original_rows": int(len(df)),
    "original_columns": int(len(df.columns)),
    "eligible_rows": int(len(model_df)),
    "home_deliveries": int(
        (model_df["facility_delivery"] == 0).sum()
    ),
    "facility_deliveries": int(
        (model_df["facility_delivery"] == 1).sum()
    ),
    "unweighted_facility_prevalence": float(
        unweighted_prevalence
    ),
    "weighted_facility_prevalence": float(
        weighted_prevalence
    ),
    "unique_psus": int(model_df["v021"].nunique()),
    "unique_v022_strata": int(model_df["v022"].nunique()),
    "v001_equals_v021": bool(
        same_cluster[
            df["v001"].notna() & df["v021"].notna()
        ].all()
    )
}

with open(
    AUDIT_DIR / "audit_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(audit_summary, f, indent=4)

print("\nSTEP 1 COMPLETED")
print("Audit outputs saved to:", AUDIT_DIR.resolve())

DATASET LOADED
Dataset shape: (41821, 5394)
Number of observations: 41821
Number of variables: 5394

All required variables are present.

OUTCOME RECONSTRUCTION
Original observations: 41821
Eligible observations: 21442
Excluded observations: 20379

Outcome distribution:
facility_delivery
0    12441
1     9001
Name: count, dtype: int64

Outcome percentages:
facility_delivery
0    58.02
1    41.98
Name: proportion, dtype: float64

REGIONAL DISTRIBUTION
 v024  sample_size  facility_deliveries  facility_prevalence
    1         3838                 1997            52.032308
    2         4498                 1169            25.989329
    3         6307                  989            15.680989
    4         2338                 1887            80.710009
    5         2050                  998            48.682927
    6         2411                 1961            81.335545

PSU AND CLUSTER CHECKS
v001 missing: 0
v021 missing: 0
v001 equals v021 for all nonmissing pairs: True
Unique v001: 1

## Construct the clean analytical dataset

In [28]:
DATA_PATH = Path(r"c:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA")

OUTPUT_DIR = Path("NDHS_FL_Research")
DATA_DIR = OUTPUT_DIR / "02_clean_datasets"

DATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD SOURCE DATA
# ------------------------------------------------------------

df,meta = pyreadstat.read_dta(
    str(DATA_PATH),
    apply_value_formats=False
)

# Preserve original row position for traceability.
df = df.copy()
df["source_row_id"] = df.index.astype(int)

# ------------------------------------------------------------
# 3. RECONSTRUCT OUTCOME
# ------------------------------------------------------------

facility_codes = [21, 22, 23, 26, 31, 36]
home_codes = [11, 12]

df["m15_1"] = pd.to_numeric(
    df["m15_1"],
    errors="coerce"
)

df["facility_delivery"] = np.select(
    [
        df["m15_1"].isin(home_codes),
        df["m15_1"].isin(facility_codes)
    ],
    [
        0,
        1
    ],
    default=np.nan
)

# Retain only observations with a valid outcome.
analytic_df = df.loc[
    df["facility_delivery"].notna()
].copy()

analytic_df["facility_delivery"] = (
    analytic_df["facility_delivery"].astype(int)
)

# ------------------------------------------------------------
# 4. DERIVE AGE AT BIRTH
# ------------------------------------------------------------

analytic_df["b3_01"] = pd.to_numeric(
    analytic_df["b3_01"],
    errors="coerce"
)

analytic_df["v011"] = pd.to_numeric(
    analytic_df["v011"],
    errors="coerce"
)

analytic_df["age_at_birth"] = np.floor(
    (analytic_df["b3_01"] - analytic_df["v011"]) / 12
)

# Check for invalid derived ages.
invalid_age = (
    analytic_df["age_at_birth"].isna()
    | (analytic_df["age_at_birth"] < 10)
    | (analytic_df["age_at_birth"] > 55)
)

print("Missing or out-of-range age_at_birth:",
      int(invalid_age.sum()))

# Do not automatically delete these records here.
# We will decide the appropriate treatment during preprocessing.

# ------------------------------------------------------------
# 5. DEFINE THE TWO CO-PRIMARY FEATURE CONFIGURATIONS
# ------------------------------------------------------------

baseline_features = [
    "age_at_birth",
    "v025",
    "v106",
    "v190",
    "v714",
    "bord_01"
]

expanded_features = baseline_features + [
    "m14_1",
    "v467d",
    "v157",
    "v158",
    "v159",
    "v701",
    "v130",
    "v131"
]

# ------------------------------------------------------------
# 6. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required = (
    baseline_features
    + [
        "m14_1", "v467d", "v157", "v158",
        "v159", "v701", "v130", "v131",
        "v024", "v001", "v021",
        "v005", "v022", "v023",
        "v501", "facility_delivery",
        "source_row_id"
    ]
)

missing = [
    col for col in required
    if col not in analytic_df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# ------------------------------------------------------------
# 7. CREATE A PROVISIONAL CATEGORICAL VERSION OF v701
# ------------------------------------------------------------

# Preserve the original numeric v701 values.
analytic_df["v701_raw"] = analytic_df["v701"]

# Retrieve DHS value labels, if present.
v701_labels = meta.variable_value_labels.get("v701", {})

# Convert known numeric codes to readable categories.
def label_v701(value):

    if pd.isna(value):
        return "Missing - classification pending"

    numeric_value = int(value)

    if numeric_value in v701_labels:
        return str(v701_labels[numeric_value])

    return f"Code {numeric_value}"


analytic_df["v701_category"] = (
    analytic_df["v701_raw"].apply(label_v701)
)

print("\nv701 category distribution:")
print(
    analytic_df["v701_category"]
    .value_counts(dropna=False)
)

# NOTE:
# "Missing - classification pending" is a provisional category.
# Do not regard it as a confirmed DHS response category.
# Review Step 1's v701 labels and marital-status cross-tabulation
# before using this variable in final model preprocessing.

# ------------------------------------------------------------
# 8. CREATE CORE ANALYTICAL DATASET
# ------------------------------------------------------------

core_columns = [
    "source_row_id",
    "facility_delivery",
    "v024"
]

core_df = analytic_df[core_columns].copy()

# ------------------------------------------------------------
# 9. CREATE BASELINE FEATURE DATASET
# ------------------------------------------------------------

baseline_columns = (
    ["source_row_id"]
    + baseline_features
    + ["facility_delivery", "v024"]
)

baseline_df = analytic_df[baseline_columns].copy()

# ------------------------------------------------------------
# 10. CREATE EXPANDED FEATURE DATASET
# ------------------------------------------------------------

expanded_columns = (
    ["source_row_id"]
    + baseline_features
    + [
        "m14_1",
        "v467d",
        "v157",
        "v158",
        "v159",
        "v130",
        "v131",
        "facility_delivery",
        "v024"
    ]
)

expanded_df = analytic_df[expanded_columns].copy()

# Use the readable provisional category for v701.
expanded_df["v701"] = analytic_df["v701_category"]

# ------------------------------------------------------------
# 11. CREATE SEPARATE SURVEY-DESIGN METADATA
# ------------------------------------------------------------

metadata_columns = [
    "source_row_id",
    "v024",
    "v001",
    "v021",
    "v005",
    "v022",
    "v023",
    "v501"
]

survey_metadata = analytic_df[metadata_columns].copy()

survey_metadata["survey_weight"] = (
    pd.to_numeric(
        survey_metadata["v005"],
        errors="coerce"
    ) / 1_000_000
)

# ------------------------------------------------------------
# 12. INTEGRITY CHECKS
# ------------------------------------------------------------

assert len(core_df) == len(baseline_df)
assert len(core_df) == len(expanded_df)
assert len(core_df) == len(survey_metadata)

assert core_df["source_row_id"].is_unique
assert baseline_df["source_row_id"].is_unique
assert expanded_df["source_row_id"].is_unique
assert survey_metadata["source_row_id"].is_unique

assert (
    core_df["source_row_id"].values
    == baseline_df["source_row_id"].values
).all()

assert (
    core_df["source_row_id"].values
    == expanded_df["source_row_id"].values
).all()

assert (
    core_df["source_row_id"].values
    == survey_metadata["source_row_id"].values
).all()

assert set(core_df["facility_delivery"].unique()) == {0, 1}

assert set(core_df["v024"].dropna().unique()) == {
    1, 2, 3, 4, 5, 6
}

assert survey_metadata["survey_weight"].notna().all()
assert (survey_metadata["survey_weight"] > 0).all()

# ------------------------------------------------------------
# 13. REPORT DATASET DIMENSIONS
# ------------------------------------------------------------

print("\nCLEAN DATASET SUMMARY")
print("=" * 60)

print("Analytical observations:", len(core_df))

print(
    "Baseline source predictors:",
    len(baseline_features)
)

print(
    "Expanded source predictors:",
    len(expanded_features)
)

print("Baseline dataset shape:", baseline_df.shape)
print("Expanded dataset shape:", expanded_df.shape)
print("Survey metadata shape:", survey_metadata.shape)

print("\nBaseline missingness:")
print(
    baseline_df[baseline_features]
    .isna()
    .sum()
)

print("\nExpanded missingness:")
print(
    expanded_df[expanded_features]
    .isna()
    .sum()
)

# ------------------------------------------------------------
# 14. SAVE DATASETS
# ------------------------------------------------------------

core_df.to_csv(
    DATA_DIR / "analytic_core.csv",
    index=False
)

baseline_df.to_csv(
    DATA_DIR / "baseline_features.csv",
    index=False
)

expanded_df.to_csv(
    DATA_DIR / "expanded_features.csv",
    index=False
)

survey_metadata.to_csv(
    DATA_DIR / "survey_design_metadata.csv",
    index=False
)

# ------------------------------------------------------------
# 15. SAVE FEATURE CONFIGURATION AND DATA MANIFEST
# ------------------------------------------------------------

manifest = {
    "study": (
        "Federated Learning Under Geographic Structural "
        "Heterogeneity Across Nigeria's Six Geopolitical Zones"
    ),
    "source_file": DATA_PATH.name,
    "analytical_sample_size": int(len(core_df)),
    "target": "facility_delivery",
    "client_assignment": "v024",
    "baseline_features": baseline_features,
    "expanded_features": expanded_features,
    "survey_metadata": [
        "v005",
        "v021",
        "v022",
        "v023",
        "v001"
    ],
    "weight_formula": "v005 / 1000000",
    "v701_status": (
        "Provisional categorical representation; "
        "missingness classification requires review."
    ),
    "preprocessing_performed": False,
    "train_test_split_performed": False,
    "model_training_performed": False
}

with open(
    DATA_DIR / "dataset_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(manifest, f, indent=4)

print("\nSTEP 2 COMPLETED")
print("Datasets saved to:", DATA_DIR.resolve())
print("\nNo train/test split or model training has been performed.")


Missing or out-of-range age_at_birth: 0

v701 category distribution:
v701_category
no education                        7121
secondary                           6857
higher                              2998
primary                             2848
Missing - classification pending    1339
don't know                           279
Name: count, dtype: int64

CLEAN DATASET SUMMARY
Analytical observations: 21442
Baseline source predictors: 6
Expanded source predictors: 14
Baseline dataset shape: (21442, 9)
Expanded dataset shape: (21442, 17)
Survey metadata shape: (21442, 9)

Baseline missingness:
age_at_birth    0
v025            0
v106            0
v190            0
v714            0
bord_01         0
dtype: int64

Expanded missingness:
age_at_birth    0
v025            0
v106            0
v190            0
v714            0
bord_01         0
m14_1           0
v467d           0
v157            0
v158            0
v159            0
v701            0
v130            0
v131            0
dtype:

In [29]:
## additional diagnostic checks for v701 missingness by marital status

# Reconstruct eligible analytical sample
facility_codes = [21, 22, 23, 26, 31, 36]
home_codes = [11, 12]

df["facility_delivery"] = np.select(
    [
        df["m15_1"].isin(home_codes),
        df["m15_1"].isin(facility_codes)
    ],
    [0, 1],
    default=np.nan
)

analytic_df = df.loc[
    df["facility_delivery"].notna()
].copy()

# Identify missing v701
analytic_df["v701_missing"] = (
    analytic_df["v701"].isna()
)

# Inspect marital status labels
print("v501 variable label:")
print(meta.column_names_to_labels.get("v501"))

print("\nv501 value labels:")
print(meta.variable_value_labels.get("v501"))

print("\nv701 variable label:")
print(meta.column_names_to_labels.get("v701"))

print("\nv701 value labels:")
print(meta.variable_value_labels.get("v701"))

# Cross-tabulation
cross_tab = pd.crosstab(
    analytic_df["v501"],
    analytic_df["v701_missing"],
    margins=True
)

print("\nMissing v701 by marital status:")
print(cross_tab)

# Percent missing within each marital status
missing_rate = (
    analytic_df.groupby("v501")["v701_missing"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

missing_rate["missing_percentage"] = (
    missing_rate["mean"] * 100
)

print("\nMissingness percentages:")
print(missing_rate.to_string(index=False))

# Save diagnostic outputs
cross_tab.to_csv("v701_marital_status_crosstab.csv")
missing_rate.to_csv(
    "v701_marital_status_missingness.csv",
    index=False
)

v501 variable label:
current marital status

v501 value labels:
{0: 'never in union', 1: 'married', 2: 'living with partner', 3: 'widowed', 4: 'divorced', 5: 'no longer living together/separated'}

v701 variable label:
husband/partner's education level

v701 value labels:
{0: 'no education', 1: 'primary', 2: 'secondary', 3: 'higher', 8: "don't know"}

Missing v701 by marital status:
v701_missing  False  True    All
v501                            
0                 0   583    583
1             19435     0  19435
2               668     0    668
3                 0   268    268
4                 0   245    245
5                 0   243    243
All           20103  1339  21442

Missingness percentages:
 v501  count  sum  mean  missing_percentage
    0    583  583   1.0               100.0
    1  19435    0   0.0                 0.0
    2    668    0   0.0                 0.0
    3    268  268   1.0               100.0
    4    245  245   1.0               100.0
    5    243  243   1.0    

In [30]:
ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)
CLEAN = ROOT / "02_clean_datasets"

expanded_path = CLEAN / "expanded_features.csv"
metadata_path = CLEAN / "survey_design_metadata.csv"

expanded = pd.read_csv(expanded_path)
metadata = pd.read_csv(metadata_path)

# Confirm that the records align using the source row identifier.
assert "source_row_id" in expanded.columns
assert "source_row_id" in metadata.columns
assert expanded["source_row_id"].is_unique
assert metadata["source_row_id"].is_unique

df = expanded.merge(
    metadata[["source_row_id", "v501"]],
    on="source_row_id",
    how="left",
    validate="one_to_one"
)

assert df["v501"].notna().all()

# Identify provisional missing labels and any remaining nulls.
missing_labels = {
    "Missing - classification pending",
    "Missing",
    "NaN"
}

is_missing_v701 = (
    df["v701"].isna()
    | df["v701"].astype(str).isin(missing_labels)
)

# Current union: v501 = 1 married, 2 living with partner.
current_union = df["v501"].isin([1, 2])

# Check the observed relationship before recoding.
assert not (is_missing_v701 & current_union).any(), (
    "Unexpected missing v701 among respondents currently in a union."
)
assert not ((~is_missing_v701) & (~current_union)).any(), (
    "Unexpected observed v701 among respondents not currently in a union."
)

df.loc[is_missing_v701, "v701"] = (
    "not applicable - no current union"
)

# Keep don't know as its own category.
assert (df["v701"] == "don't know").sum() == 279

# Save without the temporary marital-status join column.
df = df.drop(columns=["v501"])

df.to_csv(expanded_path, index=False)

print("UPDATED v701 DISTRIBUTION")
print(df["v701"].value_counts(dropna=False))

print("\nRemaining missing v701:", df["v701"].isna().sum())
print("Expanded dataset shape:", df.shape)

assert df["v701"].notna().all()
assert len(df) == 21442

decision = {
    "variable": "v701",
    "treatment": (
        "Categorical predictor; retain observed education categories, "
        "'don't know', and a distinct structural non-applicability category."
    ),
    "structural_missingness": (
        "Missing for respondents not currently in a union "
        "(v501 = 0, 3, 4, 5)."
    ),
    "observed_categories": [
        "no education",
        "primary",
        "secondary",
        "higher",
        "don't know",
        "not applicable - no current union"
    ],
    "imputation": "None at this stage",
    "encoding": "To be fitted using training data only"
}

with open(CLEAN / "v701_treatment_decision.json", "w") as f:
    json.dump(decision, f, indent=4)

print("\nDecision saved.")

UPDATED v701 DISTRIBUTION
v701
no education                         7121
secondary                            6857
higher                               2998
primary                              2848
not applicable - no current union    1339
don't know                            279
Name: count, dtype: int64

Remaining missing v701: 0
Expanded dataset shape: (21442, 17)

Decision saved.


## Creating the fixed geographic, PSU-level split (Primary Sampling Unit)

In [31]:
# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------

ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)

CLEAN = ROOT / "02_clean_datasets"
SPLIT_DIR = ROOT / "03_partitions"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_PATH = CLEAN / "baseline_features.csv"
EXPANDED_PATH = CLEAN / "expanded_features.csv"
METADATA_PATH = CLEAN / "survey_design_metadata.csv"

SPLIT_SEED = 2026
N_SPLITS = 5

# --------------------------------------------------
# 2. Load datasets
# --------------------------------------------------

baseline = pd.read_csv(BASELINE_PATH)
expanded = pd.read_csv(EXPANDED_PATH)
metadata = pd.read_csv(METADATA_PATH)

required_metadata = [
    "source_row_id", "v024", "v021", "v005",
    "v022", "v023", "survey_weight"
]

for col in required_metadata:
    assert col in metadata.columns, f"Missing metadata column: {col}"

for name, data in [
    ("baseline", baseline),
    ("expanded", expanded),
    ("metadata", metadata)
]:
    assert "source_row_id" in data.columns
    assert data["source_row_id"].is_unique, (
        f"Duplicate source_row_id in {name}"
    )

# Both feature configurations must contain exactly
# the same respondents.
baseline_ids = set(baseline["source_row_id"])
expanded_ids = set(expanded["source_row_id"])
metadata_ids = set(metadata["source_row_id"])

assert baseline_ids == expanded_ids
assert baseline_ids == metadata_ids

# Confirm target and regional assignment agree.
core = baseline[
    ["source_row_id", "facility_delivery", "v024"]
].copy()

expanded_check = expanded[
    ["source_row_id", "facility_delivery", "v024"]
].copy()

check = core.merge(
    expanded_check,
    on="source_row_id",
    suffixes=("_base", "_expanded"),
    validate="one_to_one"
)

assert (
    check["facility_delivery_base"]
    == check["facility_delivery_expanded"]
).all()

assert (
    check["v024_base"] == check["v024_expanded"]
).all()

# Attach the PSU and survey metadata.
# Firstmetadata that both sources agree on region
region_check = baseline[
                        ["source_row_id", "v024"]
                        ].merge(
                            metadata[["source_row_id", "v024"]],
                            on="source_row_id",
                            suffixes=("_features", "_metadata"),
                            validate="one_to_one"
                        )

assert(
    region_check["v024_features"] == region_check[ "v024_metadata"]
).all(), "Region mismatch between feature and meteadata files."

# Keep the region already in core
#Exclude v024 from the metadat columns being merged
metadata_columns = [
    "source_row_id",
    "v021",
    "v005",
    "v022",
    "v023",
    "survey_weight"
]

core = core.merge(
    metadata[metadata_columns],
    on="source_row_id",
    how="left",
    validate='one_to_one'
)

# confirm expected columns
print("CORE COLUMNS:")
print(core.columns.tolist())

assert "v024" in core.columns
assert "v021" in core.columns
assert core["v021"].notna().all()
assert core["v024"].notna().all()
assert core["facility_delivery"].isin([0, 1]).all()

# Each PSU must belong to exactly one region.
psu_region_counts = core.groupby("v021")["v024"].nunique()
assert (psu_region_counts == 1).all()

# --------------------------------------------------
# 3. Construct region-outcome stratification labels
# --------------------------------------------------

core["stratum"] = (
    core["v024"].astype(str)
    + "_"
    + core["facility_delivery"].astype(str)
)

# Five folds: approximately 20% each.
# PSU grouping prevents cluster leakage.
splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SPLIT_SEED
)

fold_assignment = np.full(len(core), -1, dtype=int)

for fold, (_, fold_index) in enumerate(
    splitter.split(
        X=np.zeros((len(core), 1)),
        y=core["stratum"],
        groups=core["v021"]
    )
):
    fold_assignment[fold_index] = fold

assert (fold_assignment >= 0).all()

# Fold 0 = test; fold 1 = validation;
# folds 2, 3, 4 = training.
core["partition"] = np.select(
    [
        fold_assignment == 0,
        fold_assignment == 1
    ],
    [
        "test",
        "validation"
    ],
    default="train"
)

# --------------------------------------------------
# 4. Integrity checks
# --------------------------------------------------

# No PSU may cross partitions.
psu_partition_counts = (
    core.groupby("v021")["partition"].nunique()
)

assert (psu_partition_counts == 1).all(), (
    "PSU leakage detected across partitions."
)

# All regions and both outcome classes must occur
# in every partition.
coverage = pd.crosstab(
    [core["partition"], core["v024"]],
    core["facility_delivery"]
).reindex(columns=[0, 1], fill_value=0)

print("\nREGION / OUTCOME PARTITION COVERAGE")
print(coverage)

assert (coverage > 0).all().all(), (
    "At least one region/outcome combination is absent "
    "from a partition. Do not proceed."
)

# --------------------------------------------------
# 5. Partition summaries
# --------------------------------------------------

summary = (
    core.groupby(["partition", "v024"])
    .agg(
        respondents=("source_row_id", "count"),
        home_deliveries=(
            "facility_delivery",
            lambda x: int((x == 0).sum())
        ),
        facility_deliveries=(
            "facility_delivery",
            lambda x: int((x == 1).sum())
        ),
        psus=("v021", "nunique"),
        weighted_population=("survey_weight", "sum")
    )
    .reset_index()
)

summary["facility_prevalence"] = (
    summary["facility_deliveries"]
    / summary["respondents"]
)

print("\nPARTITION SUMMARY")
print(summary.to_string(index=False))

print("\nPARTITION TOTALS")
print(core["partition"].value_counts())

print("\nUNIQUE PSUs")
print(core.groupby("partition")["v021"].nunique())

# --------------------------------------------------
# 6. Save a single authoritative partition assignment
# --------------------------------------------------

assignments = core[
    ["source_row_id", "v024", "v021", "facility_delivery",
     "partition"]
].copy()

assignments.to_csv(
    SPLIT_DIR / "fixed_partition_assignments.csv",
    index=False
)

# Apply identical respondent assignments to both datasets.
for name, data in [
    ("baseline", baseline),
    ("expanded", expanded)
]:
    output = data.merge(
        assignments[["source_row_id", "partition"]],
        on="source_row_id",
        how="left",
        validate="one_to_one"
    )

    assert output["partition"].notna().all()

    output.to_csv(
        SPLIT_DIR / f"{name}_partitioned.csv",
        index=False
    )

# --------------------------------------------------
# 7. Save split manifest
# --------------------------------------------------

manifest = {
    "dataset": "2018 Nigeria Demographic and Health Survey",
    "analytical_observations": int(len(core)),
    "split_method": "StratifiedGroupKFold",
    "stratification": "geopolitical zone x facility_delivery",
    "grouping_variable": "v021",
    "n_splits": N_SPLITS,
    "split_seed": SPLIT_SEED,
    "fold_assignment": {
        "test": 0,
        "validation": 1,
        "train": [2, 3, 4]
    },
    "intended_proportions": {
        "train": 0.60,
        "validation": 0.20,
        "test": 0.20
    },
    "feature_configurations_share_assignments": True,
    "psu_leakage_check_passed": True,
    "region_outcome_coverage_check_passed": True,
    "model_training_performed": False
}

with open(SPLIT_DIR / "partition_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

summary.to_csv(
    SPLIT_DIR / "partition_summary_by_region.csv",
    index=False
)

print("\nSTEP 3 COMPLETED")
print("Saved to:", SPLIT_DIR)
print("No model training or preprocessing has been performed.")


CORE COLUMNS:
['source_row_id', 'facility_delivery', 'v024', 'v021', 'v005', 'v022', 'v023', 'survey_weight']

REGION / OUTCOME PARTITION COVERAGE
facility_delivery     0     1
partition  v024              
test       1        367   399
           2        665   232
           3       1075   196
           4         90   376
           5        211   200
           6         89   392
train      1       1105  1199
           2       1997   701
           3       3174   593
           4        271  1131
           5        632   599
           6        272  1177
validation 1        369   399
           2        667   236
           3       1069   200
           4         90   380
           5        209   199
           6         89   392

PARTITION SUMMARY
 partition  v024  respondents  home_deliveries  facility_deliveries  psus  weighted_population  facility_prevalence
      test     1          766              367                  399    49           691.422458             0.520888
  

## Auditing features before preprocessing

In [32]:
SPLIT_DIR = ROOT / "03_partitions"
AUDIT_DIR = ROOT / "04_preprocessing_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "baseline": pd.read_csv(
        SPLIT_DIR / "baseline_partitioned.csv"
    ),
    "expanded": pd.read_csv(
        SPLIT_DIR / "expanded_partitioned.csv"
    )
}

baseline_features = [
    "age_at_birth", "v025", "v106",
    "v190", "v714", "bord_01"
]

expanded_features = baseline_features + [
    "m14_1", "v467d", "v157", "v158",
    "v159", "v701", "v130", "v131"
]

feature_sets = {
    "baseline": baseline_features,
    "expanded": expanded_features
}

# Variables that should be treated as categorical,
# even if represented by numeric codes.
categorical_features = [
    "v025", "v106", "v190", "v714", "bord_01",
    "v467d", "v157", "v158", "v159",
    "v701", "v130", "v131"
]

audit_records = []
category_records = []
partition_missingness = []

for name, df in datasets.items():

    features = feature_sets[name]

    print("\n" + "=" * 65)
    print(name.upper(), "FEATURE AUDIT")
    print("=" * 65)

    assert set(features).issubset(df.columns)
    assert df["source_row_id"].is_unique
    assert df["partition"].isin(
        ["train", "validation", "test"]
    ).all()

    for col in features:

        series = df[col]
        is_categorical = col in categorical_features

        record = {
            "configuration": name,
            "feature": col,
            "dtype": str(series.dtype),
            "measurement_type": (
                "categorical" if is_categorical else "continuous"
            ),
            "missing_total": int(series.isna().sum()),
            "unique_values": int(series.nunique(dropna=True))
        }

        if pd.api.types.is_numeric_dtype(series):
            record["minimum"] = (
                float(series.min()) if series.notna().any() else None
            )
            record["maximum"] = (
                float(series.max()) if series.notna().any() else None
            )
            record["mean"] = (
                float(series.mean()) if series.notna().any() else None
            )
        else:
            record["minimum"] = None
            record["maximum"] = None
            record["mean"] = None

        audit_records.append(record)

        print(f"\n{col}")
        print("dtype:", series.dtype)
        print("unique:", series.nunique(dropna=True))
        print("missing:", series.isna().sum())

        if is_categorical:
            counts = (
                series.astype("string")
                .fillna("<MISSING>")
                .value_counts(dropna=False)
            )

            print("category frequencies:")
            print(counts.head(30).to_string())

            for category, count in counts.items():
                category_records.append({
                    "configuration": name,
                    "feature": col,
                    "category": str(category),
                    "count": int(count)
                })

        else:
            print("numeric summary:")
            print(series.describe().to_string())

        # Missingness by partition.
        for partition in ["train", "validation", "test"]:
            subset = df.loc[df["partition"] == partition, col]

            partition_missingness.append({
                "configuration": name,
                "feature": col,
                "partition": partition,
                "count": int(len(subset)),
                "missing": int(subset.isna().sum()),
                "missing_percentage": (
                    float(subset.isna().mean() * 100)
                )
            })

# --------------------------------------------------
# Check category consistency across partitions
# --------------------------------------------------

unknown_category_records = []

for name, df in datasets.items():

    for col in feature_sets[name]:

        if col not in categorical_features:
            continue

        train_categories = set(
            df.loc[df["partition"] == "train", col]
            .dropna().astype(str).unique()
        )

        for partition in ["validation", "test"]:

            evaluation_categories = set(
                df.loc[df["partition"] == partition, col]
                .dropna().astype(str).unique()
            )

            unseen = evaluation_categories - train_categories

            unknown_category_records.append({
                "configuration": name,
                "feature": col,
                "partition": partition,
                "unseen_category_count": len(unseen),
                "unseen_categories": " | ".join(sorted(unseen))
            })

# --------------------------------------------------
# Save audit outputs
# --------------------------------------------------

audit_df = pd.DataFrame(audit_records)
category_df = pd.DataFrame(category_records)
missingness_df = pd.DataFrame(partition_missingness)
unseen_df = pd.DataFrame(unknown_category_records)

audit_df.to_csv(
    AUDIT_DIR / "feature_type_and_range_audit.csv",
    index=False
)

category_df.to_csv(
    AUDIT_DIR / "feature_category_frequencies.csv",
    index=False
)

missingness_df.to_csv(
    AUDIT_DIR / "feature_missingness_by_partition.csv",
    index=False
)

unseen_df.to_csv(
    AUDIT_DIR / "unseen_categories_audit.csv",
    index=False
)

print("\n" + "=" * 65)
print("MISSINGNESS BY PARTITION")
print("=" * 65)
print(missingness_df.to_string(index=False))

print("\n" + "=" * 65)
print("CATEGORIES IN VALIDATION/TEST NOT SEEN IN TRAIN")
print("=" * 65)
print(unseen_df.to_string(index=False))

print("\nSTEP 4 FEATURE AUDIT COMPLETED")
print("Saved to:", AUDIT_DIR)
print("No preprocessing or model training performed.")



BASELINE FEATURE AUDIT

age_at_birth
dtype: float64
unique: 38
missing: 0
numeric summary:
count    21442.000000
mean        27.898797
std          6.948854
min         12.000000
25%         23.000000
50%         27.000000
75%         33.000000
max         49.000000

v025
dtype: int64
unique: 2
missing: 0
category frequencies:
v025
2    13912
1     7530

v106
dtype: int64
unique: 4
missing: 0
category frequencies:
v106
0    9501
2    6850
1    3324
3    1767

v190
dtype: int64
unique: 5
missing: 0
category frequencies:
v190
1    4997
2    4865
3    4493
4    3924
5    3163

v714
dtype: int64
unique: 2
missing: 0
category frequencies:
v714
1    14525
0     6917

bord_01
dtype: int64
unique: 17
missing: 0
category frequencies:
bord_01
2     3702
1     3657
3     3263
4     2848
5     2308
6     1798
7     1358
8      998
9      626
10     429
11     248
12     127
13      50
14      18
15       6
16       5
17       1

EXPANDED FEATURE AUDIT

age_at_birth
dtype: float64
unique: 38
missi

### verifying DHS special codes

In [33]:
SOURCE = Path(r"C:\Users\usher\Downloads\NGIR7BDT\NGIR7BFL.DTA")

# Read original DHS file and metadata.
raw, meta = pyreadstat.read_dta(SOURCE)

# Reconstruct the analytical sample using the established outcome rule.
eligible_codes = [11, 12, 21, 22, 23, 26, 31, 36]

analytic = raw.loc[
    raw["m15_1"].isin(eligible_codes)
].copy()

# Exclude code 96 (Other), as in the established sample.
analytic = analytic.loc[
    analytic["m15_1"] != 96
].copy()

variables = [
    "m14_1", "v130", "v131", "v190",
    "v025", "v106", "v714", "v467d",
    "v157", "v158", "v159", "v701",
    "bord_01"
]

for var in variables:
    print("\n" + "=" * 60)
    print("VARIABLE:", var)

    print("Variable label:")
    print(meta.column_names_to_labels.get(var, "No label available"))

    print("\nOriginal value labels:")
    print(meta.variable_value_labels.get(var, "No value labels available"))

    if var in analytic.columns:
        print("\nObserved values in analytical sample:")
        print(
            analytic[var]
            .value_counts(dropna=False)
            .sort_index()
            .to_string()
        )

print("\nDIAGNOSTIC COMPLETED")



VARIABLE: m14_1
Variable label:
number of antenatal visits during pregnancy

Original value labels:
{0: 'no antenatal visits', 98: "don't know"}

Observed values in analytical sample:
m14_1
0     5307
1      578
2      954
3     2217
4     2688
5     2309
6     2049
7     1041
8     1049
9      436
10     784
11     156
12     461
13      79
14     111
15     215
16     162
17      56
18     111
19      33
20     332
98     314

VARIABLE: v130
Variable label:
religion

Original value labels:
{1: 'catholic', 2: 'other christian', 3: 'islam', 4: 'traditionalist', 96: 'other'}

Observed values in analytical sample:
v130
1      1873
2      6770
3     12625
4        72
96      102

VARIABLE: v131
Variable label:
ethnicity

Original value labels:
{1: 'ekoi', 2: 'fulani', 3: 'hausa', 4: 'ibibio', 5: 'igala', 6: 'igbo', 7: 'ijaw/izon', 8: 'kanuri/beriberi', 9: 'tiv', 10: 'yoruba', 96: 'other', 98: "don't know"}

Observed values in analytical sample:
v131
1      108
2     1981
3     6754
4    

## fitting and validating the preprocessing pipelines

In [34]:
# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

ROOT = Path(
    r"C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research"
)

SPLIT_DIR = ROOT / "03_partitions"
OUTPUT_DIR = ROOT / "05_preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARTITIONS = ["train", "validation", "test"]

SEED = 2026

# ============================================================
# 2. FEATURE DEFINITIONS
# ============================================================

BASELINE_CATEGORICAL = [
    "v025", "v106", "v190", "v714"
]

BASELINE_NUMERIC = [
    "age_at_birth", "bord_01"
]

EXPANDED_CATEGORICAL = BASELINE_CATEGORICAL + [
    "v467d", "v157", "v158", "v159",
    "v701", "v130", "v131"
]

# m14_1 is handled separately because code 98 means
# don't know, not 98 antenatal visits.
EXPANDED_NUMERIC = [
    "age_at_birth", "bord_01", "m14_1"
]

TARGET = "facility_delivery"
METADATA_COLUMNS = [
    "source_row_id", "v024", "partition"
]

# ============================================================
# 3. LOAD PARTITIONED DATASETS
# ============================================================

baseline = pd.read_csv(
    SPLIT_DIR / "baseline_partitioned.csv"
)

expanded = pd.read_csv(
    SPLIT_DIR / "expanded_partitioned.csv"
)

datasets = {
    "baseline": baseline,
    "expanded": expanded
}

# Verify common respondents and fixed assignments.
for name, df in datasets.items():

    required = (
        METADATA_COLUMNS
        + [TARGET]
    )

    assert set(required).issubset(df.columns)
    assert df["source_row_id"].is_unique
    assert df["partition"].isin(PARTITIONS).all()
    assert df[TARGET].isin([0, 1]).all()

baseline_assignments = (
    baseline[["source_row_id", "partition"]]
    .sort_values("source_row_id")
    .reset_index(drop=True)
)

expanded_assignments = (
    expanded[["source_row_id", "partition"]]
    .sort_values("source_row_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    baseline_assignments,
    expanded_assignments
)

assert set(baseline["source_row_id"]) == set(
    expanded["source_row_id"]
)

# ============================================================
# 4. PREPARE THE m14_1 UNKNOWN INDICATOR
# ============================================================

expanded = expanded.copy()

expanded["m14_1"] = pd.to_numeric(
    expanded["m14_1"],
    errors="raise"
)

# Confirm only valid counts or the DHS don't-know code.
valid_m14 = (
    expanded["m14_1"].between(0, 20)
    | expanded["m14_1"].eq(98)
)

assert valid_m14.all(), (
    "Unexpected m14_1 values found. Inspect before proceeding."
)

# Separate indicator for don't-know responses.
expanded["m14_1_unknown"] = (
    expanded["m14_1"].eq(98).astype(int)
)

# 98 is not a count; replace it with missing.
expanded.loc[
    expanded["m14_1"].eq(98), "m14_1"
] = np.nan

assert expanded["m14_1_unknown"].sum() == 314

# ============================================================
# 5. NORMALIZE CATEGORICAL DATA TYPES
# ============================================================

for col in BASELINE_CATEGORICAL:
    baseline[col] = baseline[col].astype("string")

for col in EXPANDED_CATEGORICAL:
    expanded[col] = expanded[col].astype("string")

# The v701 categories must remain distinct.
expected_v701 = {
    "no education",
    "primary",
    "secondary",
    "higher",
    "don't know",
    "not applicable - no current union"
}

assert set(expanded["v701"].dropna().unique()) == expected_v701

# ============================================================
# 6. BUILD PREPROCESSING PIPELINE
# ============================================================

def make_one_hot_encoder():

    # Support both newer and older sklearn versions.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32
        )


def build_preprocessor(
    numeric_features,
    categorical_features,
    include_m14_unknown=False
):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder())
    ])

    transformers = [
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]

    if include_m14_unknown:
        transformers.append(
            ("m14_unknown", "passthrough", ["m14_1_unknown"])
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True
    )


# ============================================================
# 7. FIT ON TRAIN ONLY; TRANSFORM VALIDATION AND TEST
# ============================================================

configurations = {
    "baseline": {
        "data": baseline,
        "numeric": BASELINE_NUMERIC,
        "categorical": BASELINE_CATEGORICAL,
        "include_m14_unknown": False
    },
    "expanded": {
        "data": expanded,
        "numeric": EXPANDED_NUMERIC,
        "categorical": EXPANDED_CATEGORICAL,
        "include_m14_unknown": True
    }
}

manifest = {
    "dataset": "2018 NDHS Individual Recode",
    "split_seed": SEED,
    "preprocessing_fit_partition": "train",
    "validation_used_for_fitting": False,
    "test_used_for_fitting": False,
    "target": TARGET,
    "configurations": {},
    "sklearn_version": sklearn.__version__
}

for config_name, config in configurations.items():

    print("\n" + "=" * 65)
    print("PREPROCESSING:", config_name.upper())
    print("=" * 65)

    df = config["data"]

    feature_columns = (
        config["numeric"]
        + config["categorical"]
    )

    if config["include_m14_unknown"]:
        feature_columns += ["m14_1_unknown"]

    # Partition views.
    train_df = df.loc[
        df["partition"] == "train"
    ].copy()

    validation_df = df.loc[
        df["partition"] == "validation"
    ].copy()

    test_df = df.loc[
        df["partition"] == "test"
    ].copy()

    assert len(train_df) > 0
    assert len(validation_df) > 0
    assert len(test_df) > 0

    # Build and fit the preprocessor using training data only.
    preprocessor = build_preprocessor(
        numeric_features=config["numeric"],
        categorical_features=config["categorical"],
        include_m14_unknown=config["include_m14_unknown"]
    )

    X_train = preprocessor.fit_transform(
        train_df[feature_columns]
    )

    # Apply the already-fitted transformation.
    X_validation = preprocessor.transform(
        validation_df[feature_columns]
    )

    X_test = preprocessor.transform(
        test_df[feature_columns]
    )

    # Convert to float32 for PyTorch compatibility.
    X_train = np.asarray(X_train, dtype=np.float32)
    X_validation = np.asarray(
        X_validation, dtype=np.float32
    )
    X_test = np.asarray(X_test, dtype=np.float32)

    y_train = train_df[TARGET].to_numpy(dtype=np.float32)
    y_validation = validation_df[TARGET].to_numpy(
        dtype=np.float32
    )
    y_test = test_df[TARGET].to_numpy(dtype=np.float32)

    # --------------------------------------------------------
    # Integrity checks
    # --------------------------------------------------------

    assert X_train.shape[1] == X_validation.shape[1]
    assert X_train.shape[1] == X_test.shape[1]

    assert np.isfinite(X_train).all()
    assert np.isfinite(X_validation).all()
    assert np.isfinite(X_test).all()

    assert set(np.unique(y_train)) == {0.0, 1.0}
    assert set(np.unique(y_validation)) == {0.0, 1.0}
    assert set(np.unique(y_test)) == {0.0, 1.0}

    feature_names = (
        preprocessor.get_feature_names_out().tolist()
    )

    assert len(feature_names) == X_train.shape[1]
    assert len(feature_names) == len(set(feature_names))

    # --------------------------------------------------------
    # Save fitted preprocessor
    # --------------------------------------------------------

    joblib.dump(
        preprocessor,
        OUTPUT_DIR / f"{config_name}_preprocessor.joblib"
    )

    # Save transformed arrays and respondent identifiers.
    for partition_name, partition_df, X, y in [
        ("train", train_df, X_train, y_train),
        ("validation", validation_df, X_validation, y_validation),
        ("test", test_df, X_test, y_test)
    ]:

        np.savez_compressed(
            OUTPUT_DIR / f"{config_name}_{partition_name}.npz",
            X=X,
            y=y,
            source_row_id=partition_df[
                "source_row_id"
            ].astype(str).to_numpy(),
            v024=partition_df["v024"].to_numpy()
        )

    # Save feature names.
    pd.DataFrame({
        "feature_index": range(len(feature_names)),
        "processed_feature": feature_names
    }).to_csv(
        OUTPUT_DIR / f"{config_name}_feature_names.csv",
        index=False
    )

    # Record configuration.
    manifest["configurations"][config_name] = {
        "input_features": feature_columns,
        "numeric_features": config["numeric"],
        "categorical_features": config["categorical"],
        "processed_input_dimension": int(X_train.shape[1]),
        "train_shape": list(X_train.shape),
        "validation_shape": list(X_validation.shape),
        "test_shape": list(X_test.shape),
        "train_class_counts": {
            "home": int((y_train == 0).sum()),
            "facility": int((y_train == 1).sum())
        },
        "validation_class_counts": {
            "home": int((y_validation == 0).sum()),
            "facility": int((y_validation == 1).sum())
        },
        "test_class_counts": {
            "home": int((y_test == 0).sum()),
            "facility": int((y_test == 1).sum())
        },
        "all_values_finite": True,
        "feature_names": feature_names
    }

    print("Training matrix:", X_train.shape)
    print("Validation matrix:", X_validation.shape)
    print("Test matrix:", X_test.shape)
    print("Processed input dimension:", X_train.shape[1])
    print("All values finite: PASS")
    print("Consistent dimensions: PASS")

# ============================================================
# 8. SAVE MANIFEST
# ============================================================

manifest["respondent_assignments_shared"] = True
manifest["model_training_performed"] = False
manifest["test_used_for_model_selection"] = False

with open(
    OUTPUT_DIR / "preprocessing_manifest.json",
    "w"
) as f:
    json.dump(manifest, f, indent=4)

print("\n" + "=" * 65)
print("STEP 5 PREPROCESSING COMPLETED")
print("=" * 65)
print("Output directory:", OUTPUT_DIR)
print("No model training has been performed.")



PREPROCESSING: BASELINE
Training matrix: (12851, 15)
Validation matrix: (4299, 15)
Test matrix: (4292, 15)
Processed input dimension: 15
All values finite: PASS
Consistent dimensions: PASS

PREPROCESSING: EXPANDED
Training matrix: (12851, 51)
Validation matrix: (4299, 51)
Test matrix: (4292, 51)
Processed input dimension: 51
All values finite: PASS
Consistent dimensions: PASS

STEP 5 PREPROCESSING COMPLETED
Output directory: C:\Users\usher\Documents\NDHS-Fl-Research\NDHS_FL_Research\05_preprocessed
No model training has been performed.


In [35]:
print(core.columns.tolist())


['source_row_id', 'facility_delivery', 'v024', 'v021', 'v005', 'v022', 'v023', 'survey_weight', 'stratum', 'partition']


In [36]:
core.columns = core.columns.str.strip()
core.columns

Index(['source_row_id', 'facility_delivery', 'v024', 'v021', 'v005', 'v022',
       'v023', 'survey_weight', 'stratum', 'partition'],
      dtype='str')